## Bước 1 — Cấu hình và kiểm tra

In [ ]:
import os, sys, json, shutil, subprocess

VOL        = "/mnt/m3diff"                       # Volume tạo ở Files panel
REPO       = "/root/M3Diff"
VENV       = "/root/m3diff-venv"
PY         = f"{VENV}/bin/python"
CKPT       = "/root/ckpt/M3Diff"                 # checkpoint 16GB, tải lại mỗi phiên (nhanh)
RESULTS    = f"{VOL}/results/OmniDiff_Real"

# A10 / L4 / A100 (>= 24GB): QUANT="none", DTYPE="bfloat16"  -> đúng bản gốc
# T4 (16GB):                 QUANT="8bit", DTYPE="float16"
QUANT = "none"
DTYPE = "bfloat16"

# Chỉ cần nếu tải Hugging Face báo lỗi 401/403: dán token vào đây
HF_TOKEN = os.environ.get("HF_TOKEN", "")

!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir(VOL):
    print(f"\n!!! Chưa thấy Volume {VOL}. Vào Files -> tạo Volume tên m3diff. Hiện /mnt có:", os.listdir("/mnt") if os.path.isdir("/mnt") else "(trống)")
else:
    os.makedirs(RESULTS, exist_ok=True)
    print("\nVolume OK:", VOL)

## Bước 2 — Tạo môi trường Python 3.10 riêng + cài thư viện

In [ ]:
%uv pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"

## Bước 3 — Tải checkpoint M³Diff (16GB) và ảnh test OmniDiff-Real

In [ ]:
dl = f'''
from huggingface_hub import snapshot_download
tok = {HF_TOKEN!r} or None
snapshot_download("IVC-liuyuan/M3Diff", local_dir="{CKPT}", token=tok)
snapshot_download("IVC-liuyuan/OmniDiff", repo_type="dataset", local_dir="{REPO}/datasets/OmniDiff",
                  allow_patterns=["Real/test/*"], token=tok)
print("Tải xong")
'''
subprocess.run([PY, "-c", dl], check=True)

with open(f"{REPO}/data/evaluation/prompt/OmniDiff_Real_test.json") as f:
    questions = json.load(f)
needed = sorted({img for q in questions for img in q["image"]})
missing = [p for p in needed if not os.path.exists(f"{REPO}/datasets/OmniDiff/{p}")]
!du -sh {CKPT}
print(f"Ảnh cần: {len(needed)} | còn thiếu: {len(missing)}", missing[:10])

## Bước 4 — Patch script eval

In [ ]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

## Bước 5 — Chạy thử 3 câu
Xem thanh tiến trình để biết mỗi câu mất bao nhiêu giây (`s/it`).

In [ ]:
smoke_q = f"{REPO}/data/evaluation/prompt/OmniDiff_Real_smoke.json"
json.dump(questions[:3], open(smoke_q, "w"), indent=2)
smoke_out = f"{REPO}/results/smoke.jsonl"
if os.path.exists(smoke_out):
    os.remove(smoke_out)

%cd {REPO}
!PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py \
  --model-path {CKPT} --question-file {smoke_q} --answers-file {smoke_out} \
  --image-folder datasets/OmniDiff --num-chunks 1 --chunk-idx 0 \
  --temperature 0 --torch-dtype {DTYPE} --no-flash-attention

if os.path.exists(smoke_out):
    for line in open(smoke_out):
        r = json.loads(line)
        print(f"\n[{r['id']}]\n{r['caption']}")

## Bước 6 — Chạy 803 câu (chạy tiếp được)
Kết quả ghi vào Volume: `/mnt/m3diff/results/OmniDiff_Real/OmniDiff_Real.jsonl`, mỗi dòng 1 câu.
Nếu kernel bị tắt giữa chừng: mở lại, chạy Bước 1 → 4 rồi chạy lại cell này, nó chỉ chạy các câu còn thiếu.

In [ ]:
answers = f"{RESULTS}/OmniDiff_Real.jsonl"

def count_done(fp):
    if not os.path.exists(fp):
        return 0
    n = 0
    for l in open(fp):
        try:
            json.loads(l); n += 1
        except Exception:
            pass
    return n

print(f"Đã có: {count_done(answers)}/{len(questions)} câu")
if count_done(answers) < len(questions):
    %cd {REPO}
    !PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py \
      --model-path {CKPT} --question-file data/evaluation/prompt/OmniDiff_Real_test.json \
      --answers-file {answers} --image-folder datasets/OmniDiff --num-chunks 1 --chunk-idx 0 \
      --temperature 0 --torch-dtype {DTYPE} --no-flash-attention
print(f"Hiện có: {count_done(answers)}/{len(questions)} câu")

## Bước 7 — Tính điểm (BLEU-4, METEOR, ROUGE-L, CIDEr, SPICE)
Bộ chấm điểm cần Java. Cell tự cài Java (qua `apt`, không được thì tải JDK bằng `install-jdk`). Lần đầu SPICE tự tải thêm tài nguyên.
Bước này không cần GPU; nếu muốn tiết kiệm, đổi Compute profile sang CPU trước khi chạy (sau đó chạy lại Bước 1 để có biến).

In [ ]:
# Bước 7 — Tính điểm (cần đã chạy Bước 1 và 2 trong phiên này)
import os, json, re, shutil, subprocess

answers = f"{RESULTS}/OmniDiff_Real.jsonl"
RUN_SPICE = False   # paper không báo SPICE cho OmniDiff-Real; bật True nếu muốn thử (cần RAM >= 32 GiB)

# 1. Java (bộ chấm COCO cần Java để tách từ + tính METEOR)
if shutil.which("java") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless",
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if shutil.which("java") is None:
    java_home = subprocess.run([PY, "-c", "import jdk; print(jdk.install('11', jre=True, path='/root/.jdk'))"],
                               capture_output=True, text=True).stdout.strip()
    os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]
print("Java:", shutil.which("java"))

# 2. Nếu chạy SPICE: bỏ cache (lỗi với Java mới) và tăng bộ nhớ Java
if RUN_SPICE:
    spice_py = subprocess.run([PY, "-c", "import pycocoevalcap.spice.spice as s; print(s.__file__)"],
                              capture_output=True, text=True).stdout.strip()
    src = open(spice_py).read()
    src = re.sub(r"\s*'-cache', cache_dir,", "", src).replace("-Xmx8G", "-Xmx24G")
    open(spice_py, "w").write(src)

# 3. Chấm điểm bằng đúng các scorer COCO mà metrics.py của repo dùng
score_py = f"""
import json, sys
sys.path.insert(0, {REPO!r})
from pathlib import Path
from metrics import load_predictions
from pycocotools.coco import COCO
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

coco = COCO({REPO!r} + "/data/evaluation/reference/OmniDiff_Real.json")
res = coco.loadRes(load_predictions(Path({answers!r})))
ids = res.getImgIds()
tok = PTBTokenizer()
gts = tok.tokenize({{i: coco.imgToAnns[i] for i in ids}})
rs = tok.tokenize({{i: res.imgToAnns[i] for i in ids}})
scorers = [(Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]), (Meteor(), ["METEOR"]),
           (Rouge(), ["ROUGE_L"]), (Cider(), ["CIDEr"])]
if {RUN_SPICE!r}:
    from pycocoevalcap.spice.spice import Spice
    scorers.append((Spice(), ["SPICE"]))
out = {{}}
for scorer, names in scorers:
    s, _ = scorer.compute_score(gts, rs)
    for n, v in zip(names, s if isinstance(s, list) else [s]):
        out[n] = float(v)
print("SCORES_JSON=" + json.dumps(out))
"""
r = subprocess.run([PY, "-c", score_py], capture_output=True, text=True, cwd=REPO)
line = [l for l in r.stdout.splitlines() if l.startswith("SCORES_JSON=")]
if not line:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise RuntimeError("Chấm điểm lỗi, xem log ở trên")
scores = json.loads(line[0].split("=", 1)[1])

paper = {"Bleu_4": 14.3, "METEOR": 18.9, "ROUGE_L": 32.9, "CIDEr": 31.3}
print(f"\n{'Chỉ số':<10}{'Bạn chạy':>10}{'Paper':>8}")
for k, v in scores.items():
    print(f"{k:<10}{v*100:>10.1f}{paper.get(k, ''):>8}")

json.dump(scores, open(f"{RESULTS}/scores.json", "w"), indent=2)
print(f"\nĐã lưu: {RESULTS}/scores.json")

## Bước 8

In [ ]:
# OmniDiff-Render — tải ảnh test + chạy model (cần đã chạy Bước 1–4 trong phiên này)
def count_done(fp):
    if not os.path.exists(fp):
        return 0
    n = 0
    for l in open(fp):
        try:
            json.loads(l); n += 1
        except Exception:
            pass
    return n

dl_render = f'''
from huggingface_hub import snapshot_download
snapshot_download("IVC-liuyuan/OmniDiff", repo_type="dataset", local_dir="{REPO}/datasets/OmniDiff",
                  allow_patterns=["Render/test/*"], token={HF_TOKEN!r} or None)
'''
subprocess.run([PY, "-c", dl_render], check=True)

render_q_file = f"{REPO}/data/evaluation/prompt/OmniDiff_Render_test.json"
render_q = json.load(open(render_q_file))
need = sorted({p for q in render_q for p in q["image"]})
miss = [p for p in need if not os.path.exists(f"{REPO}/datasets/OmniDiff/{p}")]
print(f"Ảnh Render cần: {len(need)} | còn thiếu: {len(miss)}", miss[:5])

RESULTS_RENDER = f"{VOL}/results/OmniDiff_Render"
os.makedirs(RESULTS_RENDER, exist_ok=True)
answers_render = f"{RESULTS_RENDER}/OmniDiff_Render.jsonl"
print(f"Đã có: {count_done(answers_render)}/{len(render_q)} câu")

# Script eval của Render giống hệt Real (chỉ khác đường dẫn mặc định), nên dùng lại script Real đã patch
if not miss and count_done(answers_render) < len(render_q):
    %cd {REPO}
    !PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py \
      --model-path {CKPT} --question-file {render_q_file} \
      --answers-file {answers_render} --image-folder datasets/OmniDiff --num-chunks 1 --chunk-idx 0 \
      --temperature 0 --torch-dtype {DTYPE} --no-flash-attention
print(f"Hiện có: {count_done(answers_render)}/{len(render_q)} câu")